In [ ]:
!pip install fastapi uvicorn sentence-transformers faiss-cpu \
             beautifulsoup4 requests rank_bm25 numpy aiohttp nest_asyncio pyngrok nltk

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 44.9 MB/s eta 0:00:00


In [ ]:
from google.colab import files

# This will open a file picker — upload your backend.py and frontend.html
uploaded = files.upload()

Saving backend.py to backend (1).py


In [ ]:
import nest_asyncio, uvicorn, threading
nest_asyncio.apply()

def run():
    uvicorn.run("backend:app", host="0.0.0.0", port=8000)

t = threading.Thread(target=run, daemon=True)
t.start()

from pyngrok import ngrok
ngrok.set_auth_token("3Co2UOUawa6m3LCwwvNLrWtXRjj_h78GiVaWgfS96a1ibUTb")  # from dashboard.ngrok.com
public_url = ngrok.connect(8000)
print("Your ngrok URL:", public_url)

INFO:     Started server process [7170]
INFO:     Waiting for application startup.


Batches:   0%|          | 0/5 [00:00<?, ?it/s]

Your ngrok URL: NgrokTunnel: "https://uncut-platypus-carrousel.ngrok-free.dev" -> "http://localhost:8000"


In [ ]:
# Copy the URL printed above (without quotes), paste it below
NGROK_URL =  "https://uncut-platypus-carrousel.ngrok-free.dev"

with open("frontend.html", "r") as f:
    html = f.read()

# Replace whatever URL is currently in the file
import re
html = re.sub(r'const API = ".*?"', f'const API = "{NGROK_URL}"', html)

with open("frontend_ready.html", "w") as f:
    f.write(html)

from google.colab import files
files.download("frontend_ready.html")
print("Done — open the downloaded frontend_ready.html in your browser")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Done — open the downloaded frontend_ready.html in your browser


In [ ]:
import requests, time

BACKEND = NGROK_URL  # uses the variable from Cell 4

def get_category_members(category, limit=300):
    r = requests.get("https://en.wikipedia.org/w/api.php", params={
        "action": "query", "list": "categorymembers",
        "cmtitle": f"Category:{category}",
        "cmlimit": limit, "cmtype": "page", "format": "json"
    }, headers={"User-Agent": "TheExplorer/1.0"})
    return [m["title"] for m in r.json()["query"]["categorymembers"]]

topics = (
    get_category_members("Animals", 300) +
    get_category_members("Countries", 200) +
    get_category_members("Cities", 200) +
    get_category_members("Science", 200) +
    get_category_members("Sports", 200) +
    get_category_members("History", 200) +
    get_category_members("Technology", 200) +
    get_category_members("Geography", 200) +
    get_category_members("Food and drink", 200) +
    get_category_members("Music", 200)
)

topics = list(dict.fromkeys(topics))  # deduplicate
print(f"Total topics: {len(topics)}")

r = requests.post(f"{BACKEND}/index/wikipedia", json={
    "topics": topics,
    "max_chars": 5000
})
print(r.json())

Total topics: 183
INFO:     34.148.186.143:0 - "POST /index/wikipedia HTTP/1.1" 200 OK
{'status': 'wikipedia indexing started', 'topics': 183, 'message': 'Articles are being fetched in the background. Check /stats for progress.'}


In [ ]:
while True:
    s = requests.get(f"{BACKEND}/stats").json()
    print(f"Docs indexed: {s['docs_indexed']}", end="\r")
    time.sleep(5)

INFO:     34.148.186.143:0 - "GET /stats HTTP/1.1" 200 OK
[Wiki] [98/183] ✓ 2024 Mobile Guardian security breach
[Wiki] [99/183] ✓ Android Central
[Wiki] Error fetching 'Cdial': Expecting value: line 1 column 1 (char 0)
[Wiki] [101/183] ✓ Closed-loop geothermal
INFO:     136.232.89.206:0 - "OPTIONS /health HTTP/1.1" 200 OK
[Wiki] [102/183] ✓ Criticism of technology
INFO:     136.232.89.206:0 - "GET /health HTTP/1.1" 200 OK
[Wiki] Error fetching 'Cyberpunk': Expecting value: line 1 column 1 (char 0)
[Wiki] [104/183] ✓ De-extinction
[Wiki] Error fetching 'Digital ecology': Expecting value: line 1 column 1 (char 0)
[Wiki] [106/183] ✓ Flyover (Apple Maps)
[Wiki] [107/183] ✓ Foldable smartphone
[Wiki] Error fetching 'Future technology': Expecting value: line 1 column 1 (char 0)
[Wiki] [109/183] ✓ Gerontechnology
INFO:     34.148.186.143:0 - "GET /stats HTTP/1.1" 200 OK
[Wiki] Error fetching 'Gigabit Multimedia Serial Link': Expecting value: line 1 column 1 (char 0)
[Wiki] [111/183] ✓ Histor

Batches:   0%|          | 0/5 [00:00<?, ?it/s]

INFO:     34.148.186.143:0 - "GET /stats HTTP/1.1" 200 OK
INFO:     34.148.186.143:0 - "GET /stats HTTP/1.1" 200 OK
[Explorer] Index built: 150 documents.
[Wiki] Done. 135/183 articles indexed.
INFO:     34.148.186.143:0 - "GET /stats HTTP/1.1" 200 OK


KeyboardInterrupt: 

In [ ]:
s = requests.get(f"{BACKEND}/stats").json()
print(f"Docs indexed: {s['docs_indexed']}")

JSONDecodeError: Expecting value: line 1 column 1 (char 0)